In [1]:
import sys
import time

sys.path.append("/home/alberto/UnReflectAnything/")
import torch
from rich import print

from dataset.rgbp import SCRREAM_Dataset
from utilities.visualization import rgb

%load_ext autoreload
%autoreload 2


dataset = SCRREAM_Dataset(root_dir="$DATASET_DIR/SCRREAM/", rho_s=0.6, eps=1e-8)

# Create dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

# Test loading a batch
for batch in dataloader:
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()
    }
    break


In [4]:
import main

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = main.load_and_process_config(
        config_path="config_train.yaml",
    )
model, model_distill = main.create_model_from_config(config, DEVICE)

import models
ms = models.get_model_parameter_summary(model)
print(ms["components"])
print(torch.cuda.memory_summary(device=DEVICE, abbreviated=True))
mds = models.get_model_parameter_summary(model_distill)
print(mds["components"])
print(torch.cuda.memory_summary(device=DEVICE, abbreviated=True))


MODEL    [21:20:30] Teacher Model created with 52,830,828 parameters

MODEL    [21:20:30] Distilled Model created with 38,335,980 parameters

{
    'rgb_encoder': {
        'total': 21596544,
        'trainable': 0,
        'frozen': 21596544,
        'description': 'DINOv3 backbone for RGB feature extraction'
    },
    'pol_preprocessing': {
        'total': 0,
        'trainable': 0,
        'frozen': 0,
        'description': 'Polarization preprocessing (AoLP, DoLP → [cos2θ, sin2θ, DoLP])'
    },
    'pol_encoder': {
        'total': 7393920,
        'trainable': 7393920,
        'frozen': 0,
        'description': 'POLViT encoder for polarization feature extraction'
    },
    'cross_attention': {
        'total': 7100928,
        'trainable': 7100928,
        'frozen': 0,
        'description': 'RGB-POL cross-attention fusion modules'
    },
    'specular_decoder': {
        'total': 5579812,
        'trainable': 5579812,
        'frozen': 0,
        'description': 'DPT decoder for specular component'
    },
    'diffuse_decoder': {
        'total': 5579812,
        'trainable': 5579812,
        'frozen': 0,
        'description': 'DPT decoder for diffuse component'
    },
    'highlight_decoder': {
        'total': 5579812,
        'trainable': 5579812,
        'frozen': 0,
        'description': 'DPT decoder for highlight component'
    }
}

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 409255 KiB | 689485 KiB |    944 MiB | 557526 KiB |
|---------------------------------------------------------------------------|
| Active memory         | 409255 KiB | 689485 KiB |    944 MiB | 557526 KiB |
|---------------------------------------------------------------------------|
| Requested memory      | 399294 KiB | 671053 KiB |    920 MiB | 543517 KiB |
|---------------------------------------------------------------------------|
| GPU reserved memory   | 698368 KiB | 698368 KiB |    922 MiB | 245760 KiB |
|---------------------------------------------------------------------------|
| Non-releasable memory |  43353 KiB |  92998 KiB |   1227 MiB |   1185 MiB |
|---------------------------------------------------------------------------|
| Allocations           |     664    |    1308    |    1952    |    1288    |
|---------------------------------------------------------------------------|
| Active allocs         |     664    |    1308    |    1952    |    1288    |
|---------------------------------------------------------------------------|
| GPU reserved segments |      77    |      77    |     107    |      30    |
|---------------------------------------------------------------------------|
| Non-releasable allocs |      11    |      32    |     231    |     220    |
|---------------------------------------------------------------------------|
| Oversize allocations  |       0    |       0    |       0    |       0    |
|---------------------------------------------------------------------------|
| Oversize GPU segments |       0    |       0    |       0    |       0    |
|===========================================================================|

{
    'rgb_encoder': {
        'total': 21596544,
        'trainable': 0,
        'frozen': 21596544,
        'description': 'DINOv3 backbone for RGB feature extraction'
    },
    'specular_decoder': {
        'total': 5579812,
        'trainable': 5579812,
        'frozen': 0,
        'description': 'DPT decoder for specular component'
    },
    'diffuse_decoder': {
        'total': 5579812,
        'trainable': 5579812,
        'frozen': 0,
        'description': 'DPT decoder for diffuse component'
    },
    'highlight_decoder': {
        'total': 5579812,
        'trainable': 5579812,
        'frozen': 0,
        'description': 'DPT decoder for highlight component'
    }
}

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 409255 KiB | 689485 KiB |    944 MiB | 557526 KiB |
|---------------------------------------------------------------------------|
| Active memory         | 409255 KiB | 689485 KiB |    944 MiB | 557526 KiB |
|---------------------------------------------------------------------------|
| Requested memory      | 399294 KiB | 671053 KiB |    920 MiB | 543517 KiB |
|---------------------------------------------------------------------------|
| GPU reserved memory   | 698368 KiB | 698368 KiB |    922 MiB | 245760 KiB |
|---------------------------------------------------------------------------|
| Non-releasable memory |  43353 KiB |  92998 KiB |   1227 MiB |   1185 MiB |
|---------------------------------------------------------------------------|
| Allocations           |     664    |    1308    |    1952    |    1288    |
|---------------------------------------------------------------------------|
| Active allocs         |     664    |    1308    |    1952    |    1288    |
|---------------------------------------------------------------------------|
| GPU reserved segments |      77    |      77    |     107    |      30    |
|---------------------------------------------------------------------------|
| Non-releasable allocs |      11    |      32    |     231    |     220    |
|---------------------------------------------------------------------------|
| Oversize allocations  |       0    |       0    |       0    |       0    |
|---------------------------------------------------------------------------|
| Oversize GPU segments |       0    |       0    |       0    |       0    |
|===========================================================================|

In [ ]:
from models import DINOv3, DINOv3toDPTRGB

dinov3_config = {
    'model_name': "facebook/dinov3-vits16-pretrain-lvd1689m",
    'image_size': 896,  # or any size divisible by 16
    'freeze_backbone': True,
    'return_selected_layers': [2, 5, 8, 11],  # DPT extraction points
    'return_as_feature_maps': True  # Need tokens for reassembly
}
dinov3_model = DINOv3(dinov3_config)

# Create the complete model with RGB decoder
model = DINOv3toDPTRGB(
    dinov3_model=dinov3_model,
    decoder_config={
        'use_bn': True,  # Use batch norm for training stability
        'readout_type': 'project'  # or 'project' for global context
    },
    selected_layers=[2, 5, 8, 11]  # Standard DPT layers
).cuda()

In [ ]:
from models import DINOv3, DPTRGBDecoder, POLViTEncoder, RGBPOLDecomposer

dinov3_cfg = {
    "model_name": "facebook/dinov3-vits16-pretrain-lvd1689m",
    "image_size": 448,
    "freeze_backbone": True,
    "return_last_hidden_state": True,
    "return_as_feature_maps": False,  # Need tokens for cross-attention
}

pol_enc_cfg = {
    "in_ch": 3,
    "embed_dim": 384,  # ViT-S dimension
    "depth": 4,
    "n_heads": 12,
    "patch_size": 16,
}
decS_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
decD_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
decH_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
pol_enc = POLViTEncoder(pol_enc_cfg)
dinov3 = DINOv3(dinov3_cfg)
decS = DPTRGBDecoder(decS_cfg)
decD = DPTRGBDecoder(decD_cfg)
decH = DPTRGBDecoder(decH_cfg)

model = RGBPOLDecomposer(
    dinov3=dinov3,
    pol_encoder=pol_enc,
    pol_preprocess=None,  # Use default
    pol_cross_attn=None,  # Use default
    spec_decoder=decS,
    diffuse_decoder=decD,
    highlight_decoder=decH,
).cuda()

In [ ]:
start = time.time()
decomposition = model(batch)
end = time.time()
print("RGB:", batch["rgb"].shape)
print("AoLp:", batch["AoP"].shape)
print("DoLp:", batch["DoP"].shape)
print("RGBprep:", model.dinov3.preprocess_image(batch["rgb"]).shape)
print("POLprep:", model.pol_pre(batch["AoP"], batch["DoP"]).shape)


def print_tensor_shapes(data, prefix=""):
    """Recursively print tensor names and shapes from nested dicts/lists"""
    if isinstance(data, torch.Tensor):
        print(f"{prefix}: {data.shape}")
    elif isinstance(data, dict):
        for key, value in data.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            print_tensor_shapes(value, new_prefix)
    elif isinstance(data, (list, tuple)):
        for i, item in enumerate(data):
            new_prefix = f"{prefix}[{i}]" if prefix else f"[{i}]"
            print_tensor_shapes(item, new_prefix)
    else:
        print(f"{prefix}: {type(data)} (not a tensor)")


print("Tensor Outputs:")
print_tensor_shapes(decomposition)
print(f"Time taken: {end - start} seconds")

In [ ]:
recon = (
    decomposition["specular"] + decomposition["diffuse"] + decomposition["highlight"]
)
recon.shape

In [ ]:
rgb(batch["rgb"])
rgb(model.dinov3.preprocess_image(batch["rgb"]))
rgb(batch["AoP"])
rgb(model.pol_pre(batch["AoP"], batch["AoP"]))

In [ ]:
batch["f_spec"].shape

In [ ]:
model.dinov3.processor

In [ ]:
model.pol_pre

In [ ]:
model.dinov3.processor.do_normalize = False
model.dinov3.processor.do_rescale = False

In [ ]:
import sys

sys.path.append("/home/alberto/UnReflectAnything/")
import time

import torch
import yaml
from dotmap import DotMap
from rich import print

import wandb
from dataset.scrream import SCRREAM
from losses import SSIMLoss, specular_loss
from pipelines.depth.depth import DepthPipeline
from projections import ReflectionWarp
from utilities.visualization import rgb

### Load config
CONFIG_PATH = "../config.yaml"

with open(CONFIG_PATH, "r") as f:
    config_yaml = yaml.safe_load(f)
    config_parameters = config_yaml["parameters"]
    config_training_dict = {
        k: v.get("value") for k, v in config_parameters.items() if v is not None
    }
    config = DotMap(config_training_dict)

### Load depth estimation
depthPipeline = DepthPipeline(config, model="", device="cuda")
reflection_warp = ReflectionWarp(config.IMAGE_HEIGHT, config.IMAGE_WIDTH)
reflection_warp = reflection_warp.cuda()  # Move to GPU

### Load dataset
print("Starting dataset creation")
dataset = SCRREAM(
    root_dir="/datasets/SCRREAM/", scene_names=["scene01_full_00"], rho_s=0.6, eps=1e-8
)

# Create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=config.BATCH_SIZE, shuffle=True
)

valdataset = SCRREAM(
    root_dir="/datasets/SCRREAM/", scene_names=["scene02_full_00"], rho_s=0.6, eps=1e-8
)

# Create dataloader
valdataloader = torch.utils.data.DataLoader(
    valdataset, batch_size=config.BATCH_SIZE, shuffle=True
)

### Load model
from models import DINOv3, DPTRGBDecoder, POLViTEncoder, RGBPOLDecomposer

dinov3_cfg = {
    "model_name": "facebook/dinov3-vits16-pretrain-lvd1689m",
    "image_size": 448,
    "freeze_backbone": True,
    "return_last_hidden_state": True,
    "return_as_feature_maps": False,  # Need tokens for cross-attention
}

pol_enc_cfg = {
    "in_ch": 3,
    "embed_dim": 384,  # ViT-S dimension
    "depth": 4,
    "n_heads": 12,
    "patch_size": 16,
}
decS_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
decD_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
decH_cfg = {
    "use_bn": True,
    "readout_type": "project",
    "feature_dim": 384,
    "output_image_size": (448, 448),
}
pol_enc = POLViTEncoder(pol_enc_cfg)
dinov3 = DINOv3(dinov3_cfg)
decS = DPTRGBDecoder(decS_cfg)
decD = DPTRGBDecoder(decD_cfg)
decH = DPTRGBDecoder(decH_cfg)

model = RGBPOLDecomposer(
    dinov3=dinov3,
    pol_encoder=pol_enc,
    pol_preprocess=None,  # Use default
    pol_cross_attn=None,  # Use default
    spec_decoder=decS,
    diffuse_decoder=decD,
    highlight_decoder=decH,
).cuda()

### Optimization
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
recon_loss = SSIMLoss()
spec_loss = SSIMLoss()

if not config.NO_WANDB:
    wandb.init(project="UnReflectAnything")
    wandb.watch(model, log="all")


In [ ]:
# cropper = torchvision.transforms.CenterCrop(config.IMAGE_HEIGHT)
# Test loading a batch
for epoch in range(config.EPOCHS):
    # Training loop
    model.train()
    for batch_idx, batch in enumerate(dataloader):
        start_time = time.time()
        torch.cuda.empty_cache()
        ### Set these in the config
        light_position = torch.randn((1, 3)) * config.DEPTH_SCALE_FACTOR / 2
        light_position[0, 1:] = -torch.abs(light_position[0, 1:])
        light_color = torch.tensor([1.0, 0.8, 0.8]).cuda()  # Warm light

        cropped_fspec = model.pol_pre.prep_fn(
            images=batch["f_spec"], return_tensors="pt"
        )["pixel_values"]
        cropped_rgb = model.pol_pre.prep_fn(images=batch["rgb"], return_tensors="pt")[
            "pixel_values"
        ]

        # batch["AoP"] = cropper(batch["AoP"])
        # batch["DoP"] = cropper(batch["rgb"])
        batch = {
            k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()
        }

        torch.cuda.empty_cache()
        # with torch.no_grad():
        # depth_map = depthPipeline.depth(batch["rgb"].cuda())

        # Call with point light
        # result = reflection_warp.forward_point_light(
        #     source_image=cropped_rgb.cuda(),
        #     depth_map=depth_map[0:1].cuda(),
        #     camera_intrinsics=batch["intrinsics"].cuda()[0:1],
        #     camera_pose=torch.eye(4)
        #     .unsqueeze(0)
        #     .repeat(batch["rgb"].shape[0], 1, 1)
        #     .cuda()[0:1],
        #     light_position=light_position.cuda(),
        #     light_intensity=100.0,
        #     light_color=light_color.cuda(),
        #     surface_roughness=0.1,  # Slightly rough surface
        #     reflection_strength=0.9,  # Strong reflections
        #     return_mask=True,
        #     return_artifacts=True,
        # )
        # batch["rgb_highlighted"] = result["warped"]
        # batch["highlight_masks"] = result["mask"].float().mean(dim=1, keepdim=True).bool()
        # print(f"Time taken: {time.time() - start_time} seconds")
        # print("Batch shapes:")
        # for key, tensor in batch.items():
        #     print(f"  {key}: {tensor.shape}")  # All will be [B, C, H, W]

        # print(">> Inferencing")
        # input_rgb = model.dinov3.preprocess_image(batch["rgb"].cuda())
        decomposition = model(batch)
        recon = (
            decomposition["specular"]
            + decomposition["diffuse"]
            + decomposition["highlight"]
        )
        decomposition["recon"] = recon
        # --- compute losses ---
        batch["f_spec"] = cropped_fspec
        batch["rgb"] = cropped_rgb
        losses = specular_loss(batch, decomposition, recon_loss=SSIMLoss())

        # total scalar to backprop
        lossval = losses["total"]

        # --- backward + update ---
        optimizer.zero_grad()
        lossval.backward()
        optimizer.step()
        print(f"Training loss: {lossval.item()}")
        # Log to wandb
        if not config.NO_WANDB:
            log_dict = {
                "epoch": epoch,
                "batch": batch_idx,
                "loss/train": lossval.item(),
            }

            # Log all individual losses from the losses dict
            for loss_name, loss_value in losses.items():
                if isinstance(loss_value, torch.Tensor):
                    log_dict[f"loss/{loss_name}"] = loss_value.item()
                else:
                    log_dict[f"loss/{loss_name}"] = loss_value

            # Log images every 10 batches to avoid overwhelming wandb
            if batch_idx % 4 == 0:
                # Convert tensors to numpy and ensure proper format for wandb
                original_imgs = (
                    batch["rgb"][:4].cpu().numpy()
                )  # Log first 4 images [B, C, H, W]
                reconstructed_imgs = recon[:4].detach().cpu().numpy()  # [B, C, H, W]
                f_spec_imgs = batch["f_spec"][:4].cpu().numpy()  # [B, C, H, W]
                specular_imgs = (
                    decomposition["specular"][:4].detach().cpu().numpy()
                )  # [B, C, H, W]

                # Convert from [B, C, H, W] to [B, H, W, C] for wandb
                original_imgs = original_imgs.transpose(0, 2, 3, 1)  # [B, H, W, C]
                reconstructed_imgs = reconstructed_imgs.transpose(
                    0, 2, 3, 1
                )  # [B, H, W, C]
                f_spec_imgs = f_spec_imgs.transpose(0, 2, 3, 1)  # [B, H, W, C]
                specular_imgs = specular_imgs.transpose(0, 2, 3, 1)  # [B, H, W, C]

                # Ensure values are in [0, 1] range
                original_imgs = torch.clamp(torch.from_numpy(original_imgs), 0, 1)
                reconstructed_imgs = torch.clamp(
                    torch.from_numpy(reconstructed_imgs), 0, 1
                )
                f_spec_imgs = torch.clamp(torch.from_numpy(f_spec_imgs), 0, 1)
                specular_imgs = torch.clamp(torch.from_numpy(specular_imgs), 0, 1)
                # print(original_imgs[0].shape, reconstructed_imgs[0].shape)
                log_dict.update(
                    {
                        "images/train_original": wandb.Image(original_imgs[0].numpy()),
                        "images/train_reconstructed": wandb.Image(
                            reconstructed_imgs[0].numpy()
                        ),
                        "images/train_f_spec": wandb.Image(f_spec_imgs[0].numpy()),
                        "images/train_specular": wandb.Image(specular_imgs[0].numpy()),
                    }
                )

            wandb.log(log_dict)

    # Validation loop
    # model.eval()
    # val_losses = []
    # with torch.no_grad():
    #     for val_batch_idx, val_batch in enumerate(valdataloader):
    #         torch.cuda.empty_cache()
    #         val_batch["rgb"] = cropper(val_batch["rgb"][0:1])
    #         # Generate validation depth maps
    #         val_depth_map = depthPipeline.depth(val_batch["rgb"].cuda())

    #         # Validation inference
    #         val_input_rgb = model.dinov3.preprocess_image(val_batch["rgb"].cuda())
    #         val_reconstructed = model(val_input_rgb)
    #         val_loss = loss(val_reconstructed, val_batch["rgb"].cuda())
    #         val_losses.append(val_loss.item())
    #         print(f"Validation loss: {val_loss.item()}")

    #         # Log validation images for first batch only
    #         if val_batch_idx == 0 and not config.NO_WANDB:
    #             val_original_imgs = val_batch["rgb"][:4].cpu().numpy()  # [B, C, H, W]
    #             val_reconstructed_imgs = val_reconstructed[:4].detach().cpu().numpy()  # [B, C, H, W]

    #             # Convert from [B, C, H, W] to [B, H, W, C] for wandb
    #             val_original_imgs = val_original_imgs.transpose(0, 2, 3, 1)  # [B, H, W, C]
    #             val_reconstructed_imgs = val_reconstructed_imgs.transpose(0, 2, 3, 1)  # [B, H, W, C]

    #             # Ensure values are in [0, 1] range
    #             val_original_imgs = torch.clamp(torch.from_numpy(val_original_imgs), 0, 1)
    #             val_reconstructed_imgs = torch.clamp(torch.from_numpy(val_reconstructed_imgs), 0, 1)

    #             val_log_dict = {
    #                 "epoch": epoch,
    #                 "images/val_original": wandb.Image(val_original_imgs[0].numpy()),
    #                 "images/val_reconstructed": wandb.Image(val_reconstructed_imgs[0].numpy()),
    #             }
    #             wandb.log(val_log_dict)

    # # Log average validation loss
    # avg_val_loss = sum(val_losses) / len(val_losses)
    # print(f"Epoch {epoch}: Average validation loss: {avg_val_loss:.6f}")

    # if not config.NO_WANDB:
    #     wandb.log({
    #         "epoch": epoch,
    #         "loss/val": avg_val_loss,
    #     })